# 📊 Notebook 1: Data Exploration

**Hindi Named Entity Recognition: Traditional vs Transformer-Based Approaches**

This notebook:
1. Loads the HiNER dataset (with WikiANN Hindi fallback)
2. Computes dataset statistics
3. Visualizes entity distribution, class imbalance, and script distribution
4. Shows sample sentences with BIO tags
5. Analyzes sentence length distribution

In [1]:
import sys
sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from collections import Counter

from src.data_utils import (
    load_hiner_dataset,
    compute_dataset_statistics,
    print_statistics,
    ID_TO_LABEL,
    LABEL_LIST,
    get_script_type,
)

# Plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

ModuleNotFoundError: No module named 'src'

## 1. Load Dataset

In [ ]:
dataset = load_hiner_dataset()
print(f"\nDataset keys: {list(dataset.keys())}")
print(f"\nSample example:")
print(dataset['train'][0])

## 2. Dataset Statistics

In [ ]:
# Compute statistics for each split
stats = {}
for split_name in ['train', 'validation', 'test']:
    if split_name in dataset:
        stats[split_name] = compute_dataset_statistics(dataset[split_name])
        print_statistics(stats[split_name], split_name)
    else:
        print(f"⚠ Split '{split_name}' not found in dataset")

## 3. Entity Distribution

In [ ]:
# Entity distribution bar chart
train_stats = stats.get('train', list(stats.values())[0])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Entity counts (unique entities by B- tags)
entities = train_stats['entity_counts']
colors = ['#4C72B0', '#55A868', '#C44E52']
bars = axes[0].bar(entities.keys(), entities.values(), color=colors, alpha=0.85)
axes[0].set_title('Entity Counts (Training Set)', fontweight='bold')
axes[0].set_ylabel('Count')
for bar, val in zip(bars, entities.values()):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                 f'{val:,}', ha='center', va='bottom', fontsize=10)

# Plot 2: Label distribution (all tokens including O)
label_dist = train_stats['label_distribution']
label_colors = {
    'O': '#999999',
    'B-PER': '#4C72B0', 'I-PER': '#7BA3D4',
    'B-LOC': '#55A868', 'I-LOC': '#88CC99',
    'B-ORG': '#C44E52', 'I-ORG': '#E08888',
}
label_keys = [l for l in LABEL_LIST if l in label_dist]
label_vals = [label_dist[l] for l in label_keys]
bar_colors = [label_colors.get(l, '#999999') for l in label_keys]

axes[1].bar(label_keys, label_vals, color=bar_colors, alpha=0.85)
axes[1].set_title('Token-level Label Distribution', fontweight='bold')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../results/figures/entity_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

# Show class imbalance
o_count = label_dist.get('O', 0)
entity_count = sum(v for k, v in label_dist.items() if k != 'O')
print(f"\n📊 Class Imbalance:")
print(f"   O tokens:      {o_count:>10,} ({o_count/(o_count+entity_count)*100:.1f}%)")
print(f"   Entity tokens:  {entity_count:>10,} ({entity_count/(o_count+entity_count)*100:.1f}%)")
print(f"   Ratio:          {o_count/entity_count:.1f}:1")

## 4. Script Distribution

In [ ]:
# Script distribution (Devanagari vs Latin vs other)
script_dist = train_stats['script_distribution']

fig, ax = plt.subplots(figsize=(7, 5))
script_colors = {
    'devanagari': '#FF6B35',
    'latin': '#004E89',
    'numeric': '#1A936F',
    'other': '#999999'
}
s_keys = list(script_dist.keys())
s_vals = list(script_dist.values())
s_colors = [script_colors.get(k, '#999999') for k in s_keys]

ax.pie(s_vals, labels=s_keys, colors=s_colors, autopct='%1.1f%%',
       startangle=90, textprops={'fontsize': 11})
ax.set_title('Script Distribution of Tokens', fontweight='bold', fontsize=13)

plt.tight_layout()
plt.savefig('../results/figures/script_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Sentence Length Distribution

In [ ]:
# Sentence length histogram
split_name = 'train' if 'train' in dataset else list(dataset.keys())[0]
sent_lengths = [len(ex['tokens']) for ex in dataset[split_name]]

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(sent_lengths, bins=50, color='#4C72B0', alpha=0.75, edgecolor='white')
ax.axvline(np.mean(sent_lengths), color='red', linestyle='--', 
           label=f'Mean: {np.mean(sent_lengths):.1f}')
ax.axvline(np.median(sent_lengths), color='orange', linestyle='--',
           label=f'Median: {np.median(sent_lengths):.1f}')
ax.set_xlabel('Sentence Length (tokens)')
ax.set_ylabel('Frequency')
ax.set_title('Sentence Length Distribution (Training Set)', fontweight='bold')
ax.legend()

plt.tight_layout()
plt.savefig('../results/figures/sentence_lengths.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Sample Sentences with BIO Tags

In [ ]:
# Display some sample sentences with their NER tags
print("=" * 70)
print("SAMPLE SENTENCES WITH BIO TAGS")
print("=" * 70)

# Find examples that contain entities (not all O)
interesting_examples = []
for ex in dataset[split_name]:
    if any(tag != 0 for tag in ex['ner_tags']):
        interesting_examples.append(ex)
    if len(interesting_examples) >= 10:
        break

for idx, ex in enumerate(interesting_examples[:5]):
    print(f"\n--- Example {idx+1} ---")
    print(f"Sentence: {' '.join(ex['tokens'])}")
    print(f"{'Token':20s} {'Tag':10s}")
    print("-" * 35)
    for token, tag_id in zip(ex['tokens'], ex['ner_tags']):
        label = ID_TO_LABEL.get(tag_id, f'UNK-{tag_id}')
        marker = '  ' if label == 'O' else '→ '
        print(f"{marker}{token:20s} {label:10s}")
    print()

## 7. Summary Statistics Table

This table will go into your report.

In [ ]:
# Build summary table for report
summary_rows = []
for split_name, s in stats.items():
    summary_rows.append({
        'Split': split_name.capitalize(),
        'Sentences': f"{s['num_sentences']:,}",
        'Tokens': f"{s['num_tokens']:,}",
        'Avg Length': s['avg_sentence_length'],
        'PER': f"{s['entity_counts']['PER']:,}",
        'LOC': f"{s['entity_counts']['LOC']:,}",
        'ORG': f"{s['entity_counts']['ORG']:,}",
    })

summary_df = pd.DataFrame(summary_rows)
print("\n📋 Dataset Summary (for report):")
print(summary_df.to_string(index=False))

# Save as CSV for LaTeX
summary_df.to_csv('../results/tables/dataset_summary.csv', index=False)
print("\n✓ Saved to results/tables/dataset_summary.csv")